# PPG 自适应滤波批处理实验 Notebook

本 notebook 用于按运动类型批量核验 PPG 心率估计流程，包括数据发现、质量检查、全局 Tdelay 静息段对齐诊断、未对齐 PPG-HR 测试图、Optuna 参数搜索、Stage-6 记录检查、Stage-7 replay、Stage-8 窗口诊断和 Stage-9 跨运动类型汇总。

重要约定：

- `TW_F` 是固定实验参数，只通过 `trial_param_overrides` 传入，不进入 Optuna/Bayes 搜索空间。
- `Alignment_TW` 只用于静息段 PPG-HR 提取和全局 Tdelay 搜索，不等同于训练窗口 `TW`。
- 不使用参考 HR 决定 `final_hr_bpm` 的融合选择；参考 HR 只用于评价和诊断对齐。
- 若心率曲线异常，建议先看第 3、4、6、10、11 部分的诊断图，再调整第 0 单元中的固定参数。


## 0. 初始化路径、依赖和全局默认参数

集中定义项目路径、导入当前协议 API，并固定本轮 notebook 工作流的非搜索参数。


In [ ]:
from pathlib import Path
import json
import math
import sys

import numpy as np
import pandas as pd
from IPython.display import Image, display

# 项目路径。若仓库移动，只需要同步修改 PROJECT_ROOT。
PROJECT_ROOT = Path(r"D:\python_notebook_base")
SRC_DIR = PROJECT_ROOT / "src"
TESTDATA_DIR = PROJECT_ROOT / "testdata"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"

assert PROJECT_ROOT.exists(), f"项目根目录不存在: {PROJECT_ROOT}"
assert SRC_DIR.exists(), f"源码目录不存在: {SRC_DIR}"
assert TESTDATA_DIR.exists(), f"测试数据目录不存在: {TESTDATA_DIR}"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

try:
    import optuna
    print("optuna", optuna.__version__)
except ModuleNotFoundError:
    optuna = None
    print("未安装 optuna，训练时会使用项目内置 fallback。")

from ppg_hr.experimental.alignment import align_ppg_to_ref_hr
from ppg_hr.experimental.batch_pairing import discover_sample_pairs_with_unpaired
from ppg_hr.experimental.preprocess_protocol import load_and_preprocess_protocol, resample_protocol_dataset
from ppg_hr.experimental.qc import quality_filter_sample
from ppg_hr.experimental.protocol_outputs import (
    plot_rest_alignment_diagnostics_by_motion_type,
    plot_raw_ppg_and_unaligned_hr_by_motion_type,
    plot_unaligned_fullfield_ppg_hr_by_motion_type,
)
from ppg_hr.experimental.protocol_search_space import ProtocolTrialParams
from ppg_hr.experimental.run_batch_protocol import (
    build_cross_motion_summary_table,
    build_output_run_name,
    plot_window_diagnostics_from_records,
    replay_best_record_hr_curves,
    run_batch_adaptive_protocol,
    safe_prepare_output_dir,
)
from ppg_hr.experimental.segmentation import detect_activity_segments
from ppg_hr.params import CascadeScheme, ProtocolSearchParams, TargetScope

# 全局固定参数。
FS_ORIGIN = 100
RANDOM_STATE = 42
TW_F = 0.0
NORMALIZATION_MODE = "minmax"
QC_POLICY = "fallback_baseline"
ALIGNMENT_TW = 8.0
ALIGNMENT_STEP_S = 1.0
OPTIMIZATION_OBJECTIVE = "accuracy"
DATA_SPLIT_MODE = "all_train"
DELAY_ESTIMATION_MODE = "envelope"
ENABLE_TIME_BIAS_AFTER = True
TIME_BIAS_AFTER_RANGE_S = (-5.0, 5.0)
TIME_BIAS_AFTER_STEP_S = 1.0
RECOVERY_GRACE_S = 10.0
RECOVERY_DIFF_BPM = 20.0
RECOVERY_CROSS_DIFF_BPM = 8.0

CLEAN_OUTPUTS = False
VAL_GROUPS_PER_TYPE = 1
TEST_GROUPS_PER_TYPE = 1
REST_ALIGNMENT_SCORE_MODE = "aae"
TIME_BIAS_AFTER_MODE = "posthoc_oracle_alignment"

# 静息段 PPG-HR 后处理参数。PPG/HF/CF/ACC 的分类型带通在源码中保持原有配置。
REST_HR_BAND_BPM = (40.0, 180.0)
REST_HR_BAND_HZ = tuple(v / 60.0 for v in REST_HR_BAND_BPM)
REST_HR_TRACK_BAND_BPM = 30.0
REST_HR_SLEW_LIMIT_BPM = 4.0
REST_HR_SLEW_STEP_BPM = 2.0
REST_HR_SMOOTH_METHOD = "median"
REST_HR_SMOOTH_WIN = 7
REST_HR_PEAK_PERCENT = 0.3
REST_HR_SPEC_PENALTY_ENABLE = True
REST_HR_SPEC_PENALTY_WEIGHT = 0.2
REST_HR_SPEC_PENALTY_WIDTH_HZ = 0.2

# TW_F 是固定实验参数，不进入 Optuna/Bayes 搜索空间。
# Alignment_TW 只用于静息段全局 Tdelay，不要误用为训练窗口 TW。
REST_HR_KWARGS = {
    "hr_band_bpm": REST_HR_BAND_BPM,
    "track_band_bpm": REST_HR_TRACK_BAND_BPM,
    "slew_limit_bpm": REST_HR_SLEW_LIMIT_BPM,
    "slew_step_bpm": REST_HR_SLEW_STEP_BPM,
    "smooth_method": REST_HR_SMOOTH_METHOD,
    "smooth_win": REST_HR_SMOOTH_WIN,
    "peak_percent": REST_HR_PEAK_PERCENT,
    "spec_penalty_enable": REST_HR_SPEC_PENALTY_ENABLE,
    "spec_penalty_weight": REST_HR_SPEC_PENALTY_WEIGHT,
    "spec_penalty_width_hz": REST_HR_SPEC_PENALTY_WIDTH_HZ,
}
REST_HR_TRIAL_OVERRIDES = {
    "TW_F": TW_F,
    "normalization_mode": NORMALIZATION_MODE,
    "qc_policy": QC_POLICY,
    "delay_estimation_mode": DELAY_ESTIMATION_MODE,
    "Alignment_TW": ALIGNMENT_TW,
    "Alignment_Step": ALIGNMENT_STEP_S,
    "Rest_HR_Band_BPM": REST_HR_BAND_BPM,
    "Rest_HR_Track_Band_BPM": REST_HR_TRACK_BAND_BPM,
    "Rest_HR_Slew_Limit_BPM": REST_HR_SLEW_LIMIT_BPM,
    "Rest_HR_Slew_Step_BPM": REST_HR_SLEW_STEP_BPM,
    "Rest_HR_Smooth_Method": REST_HR_SMOOTH_METHOD,
    "Rest_HR_Smooth_Win": REST_HR_SMOOTH_WIN,
    "Rest_HR_Peak_Percent": REST_HR_PEAK_PERCENT,
    "Rest_HR_Spec_Penalty_Enable": REST_HR_SPEC_PENALTY_ENABLE,
    "Rest_HR_Spec_Penalty_Weight": REST_HR_SPEC_PENALTY_WEIGHT,
    "Rest_HR_Spec_Penalty_Width_Hz": REST_HR_SPEC_PENALTY_WIDTH_HZ,
    "Rest_Alignment_Score_Mode": REST_ALIGNMENT_SCORE_MODE,
    "Enable_Time_Bias_After": ENABLE_TIME_BIAS_AFTER,
    "Time_Bias_After_Range_S": TIME_BIAS_AFTER_RANGE_S,
    "Time_Bias_After_Step_S": TIME_BIAS_AFTER_STEP_S,
    "Time_Bias_After_Mode": TIME_BIAS_AFTER_MODE,
    "Recovery_Grace_S": RECOVERY_GRACE_S,
    "Recovery_Diff_Bpm": RECOVERY_DIFF_BPM,
    "Recovery_Cross_Diff_Bpm": RECOVERY_CROSS_DIFF_BPM,
}

# 默认训练组合。正式训练时可在第 7、8 单元覆盖。
ACTIVE_TARGET_SCOPES = ["global"] if "global" in [item.value for item in TargetScope] else ["motion_post10"]
ACTIVE_CASCADE_SCHEMES = ["HF2"]
ACTIVE_ADAPTIVE_FILTERS = ["lms"]
CASCADE_TRAIN_BUDGETS = {scheme: {"n_trials": 1, "n_repeats": 1} for scheme in ACTIVE_CASCADE_SCHEMES}


def make_unique_output_dir(
    target_scopes,
    cascade_schemes,
    adaptive_filters,
    objective_mode,
    split_mode,
    *,
    tw_f=TW_F,
):
    '''根据当前训练配置创建唯一输出目录，避免覆盖已有实验结果。'''
    base_name = build_output_run_name(
        target_scopes,
        cascade_schemes,
        adaptive_filters,
        objective_mode,
        split_mode,
        tw_f_s=float(tw_f),
    )
    run_name = base_name
    run_dir = OUTPUT_ROOT / run_name
    if not CLEAN_OUTPUTS:
        idx = 1
        while run_dir.exists() and any(run_dir.iterdir()):
            run_name = f"{base_name}__run{idx:02d}"
            run_dir = OUTPUT_ROOT / run_name
            idx += 1
    run_dir = safe_prepare_output_dir(
        project_root=PROJECT_ROOT,
        output_dir=run_dir,
        clean_outputs=CLEAN_OUTPUTS,
    )
    return run_name, run_dir


OUTPUT_RUN_NAME, RUN_OUTPUT_DIR = make_unique_output_dir(
    ACTIVE_TARGET_SCOPES,
    ACTIVE_CASCADE_SCHEMES,
    ACTIVE_ADAPTIVE_FILTERS,
    OPTIMIZATION_OBJECTIVE,
    DATA_SPLIT_MODE,
    tw_f=TW_F,
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)
print("TESTDATA_DIR:", TESTDATA_DIR)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("OUTPUT_RUN_NAME:", OUTPUT_RUN_NAME)
print("RUN_OUTPUT_DIR:", RUN_OUTPUT_DIR)
print("REST_HR_TRIAL_OVERRIDES:", REST_HR_TRIAL_OVERRIDES)


## 1. 发现样本配对并执行 QC

发现 `testdata` 中的传感器 CSV 与 `_ref.csv` 配对，执行基础 QC，并额外验证新格式 `multi_tiaosheng1.csv` 的清洗后字段与 QC/元数据列。


In [ ]:
discovery = discover_sample_pairs_with_unpaired(TESTDATA_DIR)


def _pairs_to_frame(pairs):
    return pd.DataFrame([
        {
            "group_id": p.motion_id,
            "motion_type": p.motion_type,
            "motion_index": p.motion_index,
            "data_file": p.sensor_csv.name,
            "ref_file": p.ref_csv.name,
        }
        for p in pairs
    ])


pairs_df = _pairs_to_frame(discovery.pairs)
unpaired_df = pd.DataFrame([
    {"file_name": u.file_name, "file_path": str(u.file_path), "reason": u.reason}
    for u in discovery.unpaired
])

qc_rows = []
good_pairs = []
bad_pairs = []
for pair in discovery.pairs:
    qc = quality_filter_sample(
        pair.sensor_csv,
        fs=FS_ORIGIN,
        group_id=pair.motion_id,
        motion_type=pair.motion_type,
        ref_csv=pair.ref_csv,
    )
    qc_rows.append(qc.to_dict())
    if qc.is_good:
        good_pairs.append(pair)
    else:
        bad_pairs.append(pair)
qc_df = pd.DataFrame(qc_rows)
good_pairs_df = _pairs_to_frame(good_pairs)
bad_pairs_df = _pairs_to_frame(bad_pairs)

print("已配对样本数:", len(discovery.pairs))
print("未配对文件数:", len(discovery.unpaired))
print("通过 QC 样本数:", len(good_pairs))
print("未通过 QC 样本数:", len(bad_pairs))
print("good_pairs:", [p.motion_id for p in good_pairs])
print("bad_pairs:", [p.motion_id for p in bad_pairs])
print("检测到的运动类型:", sorted(pairs_df["motion_type"].unique()) if not pairs_df.empty else [])

display(good_pairs_df)
display(bad_pairs_df)
display(unpaired_df)
display(qc_df)

# 新格式读取验证：优先读取 multi_tiaosheng1.csv；若未通过 QC，则退回第一个 good sample。
validation_pair = next((p for p in good_pairs if p.sensor_csv.name == "multi_tiaosheng1.csv"), None)
if validation_pair is None and good_pairs:
    validation_pair = good_pairs[0]

QC_METADATA_COLUMNS = [
    "SampleIndex",
    "Seq",
    "ValidFlag",
    "InterpFlag",
    "GapLen",
    "MissingBefore",
    "raw_missing_any",
    "raw_missing_count",
]

if validation_pair is None:
    print("没有通过 QC 的样本，无法执行新格式读取验证。")
else:
    validation_dataset = load_and_preprocess_protocol(
        validation_pair.sensor_csv,
        validation_pair.ref_csv,
        fs_origin=FS_ORIGIN,
    )
    validation_frame = validation_dataset.to_frame()
    key_columns = [
        "time_s",
        "ppg_green",
        "hf1",
        "cf1",
        "accx",
        *QC_METADATA_COLUMNS,
    ]
    existing_key_columns = [c for c in key_columns if c in validation_frame.columns]
    print("新格式读取验证样本:", validation_pair.sensor_csv.name)
    display(validation_frame.loc[:, existing_key_columns].head())
    qc_metadata_check = pd.DataFrame([
        {
            "column": column,
            "exists": column in validation_frame.columns,
            "non_null_count": int(validation_frame[column].notna().sum()) if column in validation_frame.columns else 0,
        }
        for column in QC_METADATA_COLUMNS
    ])
    display(qc_metadata_check)


## 2. 单样本预处理、分段和全局对齐预览

使用 `good_pairs[0]` 做 smoke preview：预处理、重采样、ACC 分段，以及固定 `ALIGNMENT_TW` 的全局 Tdelay 对齐。


In [ ]:
if good_pairs:
    pair = good_pairs[0]
    ds = load_and_preprocess_protocol(pair.sensor_csv, pair.ref_csv, fs_origin=FS_ORIGIN)
    ds = resample_protocol_dataset(ds, fs_target=100)

    preview_train_TW = 8
    seg = detect_activity_segments(ds.accx, ds.accy, ds.accz, ds.fs, TW=preview_train_TW)
    if not seg.is_valid:
        print(f"分段失败: {pair.stem}: {seg.reason}")
    else:
        aligned = align_ppg_to_ref_hr(
            ds,
            seg,
            TW=preview_train_TW,
            fs_target=ds.fs,
            alignment_TW=ALIGNMENT_TW,
            alignment_step_s=ALIGNMENT_STEP_S,
            rest_hr_kwargs=REST_HR_KWARGS,
            alignment_score_mode=REST_ALIGNMENT_SCORE_MODE,
        )
        preview_row = {
            "sample": pair.stem,
            "motion_type": pair.motion_type,
            "fs": ds.fs,
            "motion_start_s": aligned.segment_info.motion_start_s,
            "motion_end_s": aligned.segment_info.motion_end_s,
            "alignment_TW": aligned.alignment_info.alignment_tw_s,
            "train_TW": aligned.alignment_info.train_tw_s,
            "best_tdelay_s": aligned.alignment_info.best_tdelay_s,
            "rest_windows": len(aligned.rest_indices),
            "motion_windows": len(aligned.motion_indices),
            "recovery_windows": len(aligned.recovery_indices),
        }
        display(pd.DataFrame([preview_row]))
else:
    print("没有通过 QC 的样本，后续训练无法继续。")


## 3. 输出全局 Tdelay 静息段对齐诊断图（Alignment_TW=8）

每个运动类型输出一张静息段对齐诊断图，用于人工核查固定 `Alignment_TW=8.0` 下的全局 Tdelay。


In [ ]:
assert float(ALIGNMENT_TW) == 8.0, f"本诊断要求 ALIGNMENT_TW=8.0，当前为 {ALIGNMENT_TW}"
ALIGNMENT_DIAGNOSTIC_DIR = RUN_OUTPUT_DIR / "alignment_diagnostics"

alignment_diagnostic_datasets = {}
for pair in good_pairs:
    try:
        alignment_diagnostic_datasets[pair.motion_id] = load_and_preprocess_protocol(
            pair.sensor_csv,
            pair.ref_csv,
            fs_origin=FS_ORIGIN,
        )
    except Exception as exc:
        print(f"alignment diagnostic preprocessing failed: {pair.motion_id}: {exc}")

alignment_diagnostic_paths = plot_rest_alignment_diagnostics_by_motion_type(
    pairs=good_pairs,
    datasets=alignment_diagnostic_datasets,
    output_dir=ALIGNMENT_DIAGNOSTIC_DIR,
    fs_target=100,
    TW=int(ALIGNMENT_TW),
    alignment_TW=ALIGNMENT_TW,
    alignment_step_s=ALIGNMENT_STEP_S,
    rest_hr_kwargs=REST_HR_KWARGS,
    alignment_score_mode=REST_ALIGNMENT_SCORE_MODE,
)
print("alignment_diagnostics:", alignment_diagnostic_paths)
for image_path in alignment_diagnostic_paths.values():
    display(Image(filename=str(image_path)))


## 4. 输出全段未对齐 PPG-HR 测试图

不做全局 Tdelay 对齐，直接对完整 PPG_Green 分窗提取 PPG-HR，用于判断异常是否来自原始 PPG 主频、分段位置或后处理参数。


In [ ]:
FULLFIELD_PPG_HR_DIR = OUTPUT_ROOT / "allfield"
FULLFIELD_PPG_HR_TW = int(ALIGNMENT_TW)

fullfield_ppg_hr_datasets = dict(globals().get("alignment_diagnostic_datasets", {}))
for pair in good_pairs:
    if pair.motion_id in fullfield_ppg_hr_datasets:
        continue
    try:
        fullfield_ppg_hr_datasets[pair.motion_id] = load_and_preprocess_protocol(
            pair.sensor_csv,
            pair.ref_csv,
            fs_origin=FS_ORIGIN,
        )
    except Exception as exc:
        print(f"fullfield PPG-HR preprocessing failed: {pair.motion_id}: {exc}")

fullfield_ppg_hr_paths = plot_unaligned_fullfield_ppg_hr_by_motion_type(
    pairs=good_pairs,
    datasets=fullfield_ppg_hr_datasets,
    output_dir=FULLFIELD_PPG_HR_DIR,
    fs_target=100,
    TW=FULLFIELD_PPG_HR_TW,
    hr_band_hz=REST_HR_BAND_HZ,
    track_band_bpm=REST_HR_TRACK_BAND_BPM,
    slew_limit_bpm=REST_HR_SLEW_LIMIT_BPM,
    slew_step_bpm=REST_HR_SLEW_STEP_BPM,
    smooth_method=REST_HR_SMOOTH_METHOD,
    smooth_win=REST_HR_SMOOTH_WIN,
    peak_percent=REST_HR_PEAK_PERCENT,
    spec_penalty_enable=REST_HR_SPEC_PENALTY_ENABLE,
    spec_penalty_weight=REST_HR_SPEC_PENALTY_WEIGHT,
    spec_penalty_width_hz=REST_HR_SPEC_PENALTY_WIDTH_HZ,
)
print("fullfield_ppg_hr_paths:", fullfield_ppg_hr_paths)
for image_path in fullfield_ppg_hr_paths.values():
    display(Image(filename=str(image_path)))


## 5. 封装训练函数

封装训练预算、notebook 进度输出和统一训练入口。`TW_F`、`normalization_mode`、`qc_policy` 只作为固定参数进入 `ProtocolTrialParams`，不进入搜索空间。


In [ ]:
def make_budgets(schemes, n_trials=1, n_repeats=1):
    '''为每个 cascade scheme 生成相同的 Optuna 搜索预算。'''
    return {scheme: {"n_trials": int(n_trials), "n_repeats": int(n_repeats)} for scheme in schemes}


PROGRESS_EVERY_N_TRIALS = 10


def notebook_progress(info):
    '''Notebook 进度回调：只打印关键 trial，避免长时间训练刷屏。'''
    stage = info.get("stage", "")
    if stage == "optimization":
        trial_idx = int(info.get("trial_idx") or 0)
        trial_total = int(info.get("trial_total") or 0)
        if trial_idx == 1 or trial_idx == trial_total or trial_idx % PROGRESS_EVERY_N_TRIALS == 0:
            print(
                f"motion_type {info.get('motion_type')} | "
                f"mode {info.get('mode_idx')}/{info.get('mode_total')} | "
                f"{info.get('target_scope_value')} / {info.get('cascade_scheme')} / {info.get('adaptive_filter')} | "
                f"repeat {info.get('repeat_idx')}/{info.get('repeat_total')} | "
                f"trial {trial_idx}/{trial_total} | "
                f"AAE {info.get('aae_bpm')} | accuracy {info.get('accuracy_pct')} | "
                f"objective_mode {info.get('objective_mode')} | objective_value {info.get('objective_value')}"
            )
    elif stage == "optimization_mode":
        print(
            f"enter mode {info.get('mode_idx')}/{info.get('mode_total')}: "
            f"{info.get('motion_type')} / {info.get('target_scope_value')} / "
            f"{info.get('cascade_scheme')} / {info.get('adaptive_filter')}"
        )
    elif stage == "optimization_parallel_repeats_start":
        print(
            f"parallel repeats start: n_jobs={info.get('n_jobs')}, repeats={info.get('repeat_total')} | "
            f"{info.get('motion_type')} / {info.get('target_scope_value')} / "
            f"{info.get('cascade_scheme')} / {info.get('adaptive_filter')} / fold={info.get('fold_id')}"
        )
    elif stage == "optimization_parallel_repeat_done":
        print(
            f"repeat {info.get('repeat_current')}/{info.get('repeat_total')} done "
            f"({info.get('repeat_done')}/{info.get('repeat_total')} completed), "
            f"best={info.get('best_value'):.4f}, global_best={info.get('global_best_value'):.4f}"
        )
    elif stage in {"qc", "preprocess"}:
        print(f"{stage} {info.get('current')}/{info.get('total')}: {info.get('sample')}")


def run_training_cell(
    *,
    active_target_scopes,
    active_cascade_schemes,
    active_adaptive_filters,
    cascade_train_budgets,
    optimization_objective,
    data_split_mode,
    val_groups_per_type=1,
    test_groups_per_type=1,
    delay_estimation_mode=DELAY_ESTIMATION_MODE,
    trial_param_overrides=REST_HR_TRIAL_OVERRIDES,
    tw_f=TW_F,
    normalization_mode=NORMALIZATION_MODE,
    qc_policy=QC_POLICY,
):
    '''运行一次批处理训练。'''
    fixed_trial_param_overrides = dict(REST_HR_TRIAL_OVERRIDES)
    if trial_param_overrides:
        fixed_trial_param_overrides.update(dict(trial_param_overrides))
    fixed_trial_param_overrides.update(
        {
            "TW_F": float(tw_f),
            "normalization_mode": str(normalization_mode),
            "qc_policy": str(qc_policy),
            "delay_estimation_mode": str(delay_estimation_mode),
        }
    )

    output_run_name, run_output_dir = make_unique_output_dir(
        active_target_scopes,
        active_cascade_schemes,
        active_adaptive_filters,
        optimization_objective,
        data_split_mode,
        tw_f=tw_f,
    )
    print("OUTPUT_RUN_NAME:", output_run_name)
    print("RUN_OUTPUT_DIR:", run_output_dir)
    print("trial_param_overrides:", fixed_trial_param_overrides)
    result = run_batch_adaptive_protocol(
        input_dir=TESTDATA_DIR,
        output_root=run_output_dir,
        max_iterations=1,
        num_repeats=1,
        random_state=RANDOM_STATE,
        num_seed_points=1,
        fs_origin=FS_ORIGIN,
        n_jobs=3,
        debug_mode=True,
        search_space=ProtocolSearchParams(),
        trial_param_overrides=fixed_trial_param_overrides,
        target_scopes=active_target_scopes,
        cascade_schemes=active_cascade_schemes,
        adaptive_filters=active_adaptive_filters,
        objective_mode=optimization_objective,
        data_split_mode=data_split_mode,
        delay_estimation_mode=delay_estimation_mode,
        val_groups_per_type=val_groups_per_type,
        test_groups_per_type=test_groups_per_type,
        cascade_train_budgets=cascade_train_budgets,
        project_root=PROJECT_ROOT,
        clean_outputs=CLEAN_OUTPUTS,
        verbose=True,
        on_log=print,
        progress_callback=notebook_progress,
    )
    print("run_output_dir:", result.output_root)
    print("batch_summary:", result.batch_summary_csv)
    print("final_summary:", result.final_summary_tables)
    print("motion_type_dirs:", result.motion_type_dirs)
    return result


## 6. 输出静息段原始 PPG 与 PPG-HR 双 y 轴图

每个运动类型输出静息段原始 PPG 与未对齐 PPG-HR 双 y 轴图，复用前面已加载的数据集，不存在时自动补加载。


In [ ]:
RAW_PPG_DUAL_AXIS_DIR = OUTPUT_ROOT / "allfield"
RAW_PPG_DUAL_AXIS_TW = int(ALIGNMENT_TW)

raw_ppg_dual_axis_datasets = dict(
    globals().get("fullfield_ppg_hr_datasets", globals().get("alignment_diagnostic_datasets", {}))
)
for pair in good_pairs:
    if pair.motion_id in raw_ppg_dual_axis_datasets:
        continue
    try:
        raw_ppg_dual_axis_datasets[pair.motion_id] = load_and_preprocess_protocol(
            pair.sensor_csv,
            pair.ref_csv,
            fs_origin=FS_ORIGIN,
        )
    except Exception as exc:
        print(f"raw PPG dual-axis preprocessing failed: {pair.motion_id}: {exc}")

raw_ppg_dual_axis_paths = plot_raw_ppg_and_unaligned_hr_by_motion_type(
    pairs=good_pairs,
    datasets=raw_ppg_dual_axis_datasets,
    output_dir=RAW_PPG_DUAL_AXIS_DIR,
    fs_target=100,
    fs_origin=FS_ORIGIN,
    TW=RAW_PPG_DUAL_AXIS_TW,
    hr_band_hz=REST_HR_BAND_HZ,
    track_band_bpm=REST_HR_TRACK_BAND_BPM,
    slew_limit_bpm=REST_HR_SLEW_LIMIT_BPM,
    slew_step_bpm=REST_HR_SLEW_STEP_BPM,
    smooth_method=REST_HR_SMOOTH_METHOD,
    smooth_win=REST_HR_SMOOTH_WIN,
    peak_percent=REST_HR_PEAK_PERCENT,
    spec_penalty_enable=REST_HR_SPEC_PENALTY_ENABLE,
    spec_penalty_weight=REST_HR_SPEC_PENALTY_WEIGHT,
    spec_penalty_width_hz=REST_HR_SPEC_PENALTY_WIDTH_HZ,
)
print("raw_ppg_dual_axis_paths:", raw_ppg_dual_axis_paths)
for image_path in raw_ppg_dual_axis_paths.values():
    display(Image(filename=str(image_path)))


## 7. 训练单元 1：all_train 全样本拟合上限检查，注意 TW_F 的赋值与输入

`all_train` 用于观察全样本拟合上限。默认优先使用 `global` target scope；若当前枚举不支持 `global`，退回 `motion_post10` 并打印原因。


In [ ]:
target_scope_values = [item.value for item in TargetScope]
if "global" in target_scope_values:
    ACTIVE_TARGET_SCOPES = ["global"]
else:
    ACTIVE_TARGET_SCOPES = ["motion_post10"]
    print("TargetScope 不包含 global，保留 motion_post10 作为 all_train 上限检查。")

ACTIVE_CASCADE_SCHEMES = ["ACC3"]
ACTIVE_ADAPTIVE_FILTERS = ["lms"]
CASCADE_TRAIN_BUDGETS = {"ACC3": {"n_trials": 1, "n_repeats": 1}}
OPTIMIZATION_OBJECTIVE = "accuracy"
DATA_SPLIT_MODE = "all_train"
VAL_GROUPS_PER_TYPE = 1
TEST_GROUPS_PER_TYPE = 1

result_all_train = run_training_cell(
    active_target_scopes=ACTIVE_TARGET_SCOPES,
    active_cascade_schemes=ACTIVE_CASCADE_SCHEMES,
    active_adaptive_filters=ACTIVE_ADAPTIVE_FILTERS,
    cascade_train_budgets=CASCADE_TRAIN_BUDGETS,
    optimization_objective=OPTIMIZATION_OBJECTIVE,
    data_split_mode=DATA_SPLIT_MODE,
    val_groups_per_type=VAL_GROUPS_PER_TYPE,
    test_groups_per_type=TEST_GROUPS_PER_TYPE,
    tw_f=TW_F,
    normalization_mode=NORMALIZATION_MODE,
    qc_policy=QC_POLICY,
)

for motion_type, motion_dir in result_all_train.motion_type_dirs.items():
    print("motion_type:", motion_type, motion_dir)
    for name in ["best_params_and_alignment.csv", "best_metrics.csv"]:
        path = motion_dir / name
        print(name, path.exists(), path)
        if path.exists():
            display(pd.read_csv(path).head())


## 8. 训练单元 2：完整 63 模式连通性检查

默认关闭。若开启，该单元运行 3 个 target scope x 7 个 cascade scheme x 3 个 adaptive filter，共 63 模式。

说明：如果需要纳入 `global/all`，组合会变为 4 x 7 x 3 = 84 模式，应另开全局连通性测试，不要混淆“63 模式”定义。


In [ ]:
RUN_FULL_63_MODE_TEST = False

if RUN_FULL_63_MODE_TEST:
    ACTIVE_TARGET_SCOPES = ["motion_only", "motion_recovery", "motion_post10"]
    ACTIVE_CASCADE_SCHEMES = ["ACC3", "HF2", "CF2", "HF2_CF2", "CF2_HF2", "ACC3_HF2", "HF2_ACC3"]
    ACTIVE_ADAPTIVE_FILTERS = ["lms", "volterra", "rff_lms"]
    CASCADE_TRAIN_BUDGETS = make_budgets(ACTIVE_CASCADE_SCHEMES, n_trials=1, n_repeats=1)
    OPTIMIZATION_OBJECTIVE = "accuracy"
    DATA_SPLIT_MODE = "split"
    VAL_GROUPS_PER_TYPE = 1
    TEST_GROUPS_PER_TYPE = 1
    result_full_63 = run_training_cell(
        active_target_scopes=ACTIVE_TARGET_SCOPES,
        active_cascade_schemes=ACTIVE_CASCADE_SCHEMES,
        active_adaptive_filters=ACTIVE_ADAPTIVE_FILTERS,
        cascade_train_budgets=CASCADE_TRAIN_BUDGETS,
        optimization_objective=OPTIMIZATION_OBJECTIVE,
        data_split_mode=DATA_SPLIT_MODE,
        val_groups_per_type=VAL_GROUPS_PER_TYPE,
        test_groups_per_type=TEST_GROUPS_PER_TYPE,
        tw_f=TW_F,
        normalization_mode=NORMALIZATION_MODE,
        qc_policy=QC_POLICY,
    )
else:
    print("RUN_FULL_63_MODE_TEST=False，已跳过完整 63 模式测试。")


## 9. 输出检查

检查新 Stage-6 记录结构：`best_params_and_alignment.csv`、`best_metrics.csv`、`motion_frequency_and_params.csv`、`full_report.json`。


In [ ]:
RESULT_TO_CHECK = globals().get(
    "result_full_63",
    globals().get("result_leave_one_group_out", globals().get("result_all_train", globals().get("result_split", None))),
)

STAGE6_FILES = [
    "best_params_and_alignment.csv",
    "best_metrics.csv",
    "motion_frequency_and_params.csv",
    "full_report.json",
]
STAGE6_REQUIRED_COLUMNS = [
    "final_aae_bpm",
    "final_acc_pct",
    "posthoc_final_aae_bpm",
    "posthoc_final_acc_pct",
    "TW_F",
    "normalization_mode",
    "best_tdelay_s",
    "time_bias_after_s",
    "n_windows",
    "n_valid_windows",
    "n_qc_fallback",
    "n_recovery_fallback",
]

if RESULT_TO_CHECK is None:
    print("没有可检查的训练结果，请先运行第 7 单元或手动设置 RESULT_TO_CHECK。")
else:
    print("output_root:", RESULT_TO_CHECK.output_root)
    print("qc_tables:", RESULT_TO_CHECK.qc_tables)
    print("batch_summary:", RESULT_TO_CHECK.batch_summary_csv)
    print("motion_type_dirs:", RESULT_TO_CHECK.motion_type_dirs)

    for motion_type, motion_dir in RESULT_TO_CHECK.motion_type_dirs.items():
        print("\nmotion_type:", motion_type, motion_dir)
        observed_columns = set()
        for name in STAGE6_FILES:
            path = motion_dir / name
            print(name, path.exists(), path)
            if not path.exists():
                continue
            if path.suffix.lower() == ".csv":
                df = pd.read_csv(path)
                observed_columns.update(df.columns)
                display(df.head())
            elif path.name == "full_report.json":
                report = json.loads(path.read_text(encoding="utf-8"))
                print("full_report keys:", sorted(report.keys()))
                distribution = report.get("fusion_source_distribution", {})
                print("fusion_source_distribution:", distribution)
                if distribution:
                    display(pd.DataFrame([distribution]))

        missing_columns = [column for column in STAGE6_REQUIRED_COLUMNS if column not in observed_columns]
        if missing_columns:
            print("Stage-6 记录缺少列:", missing_columns)
        else:
            print("Stage-6 必需列检查 OK")


## 10. 对单个文件的 best_params HR 曲线进行重绘

使用 Stage-7 `replay_best_record_hr_curves` 从 `best_params_and_alignment.csv` 恢复参数并重绘单文件 HR 曲线。旧 `best_params_lms.csv` 接口不再作为默认入口。


In [ ]:
RUN_STAGE7_REPLAY = True

REPLAY_SIGNAL_CSV = TESTDATA_DIR / "multi_tiaosheng1.csv"
REPLAY_REF_CSV = TESTDATA_DIR / "multi_tiaosheng1_ref.csv"
REPLAY_RESULTS_ROOT = Path(getattr(globals().get("RESULT_TO_CHECK", None), "output_root", RUN_OUTPUT_DIR))
REPLAY_OUTPUT_DIR = OUTPUT_ROOT / "manual_replay"
REPLAY_MOTION_TYPE = "tiaosheng"
REPLAY_TARGET_SCOPE = ACTIVE_TARGET_SCOPES[0] if ACTIVE_TARGET_SCOPES else "global"
REPLAY_ADAPTIVE_FILTER = "lms"
REPLAY_CASCADE_SCHEME = "ACC3"
REPLAY_ADAPTIVE_DATA_TYPE = ""
REPLAY_TW_F = TW_F

# 如需强制选择特定 split/mode，可在这里填入字符串；留空时自动读取 Stage-6 记录第一条匹配行。
REPLAY_MANUAL_SPLIT = ""
REPLAY_MANUAL_MODE = ""


def _stage6_motion_dir(results_root, motion_type):
    root = Path(results_root)
    candidates = [root / "motion_types" / motion_type, root / motion_type, root]
    for item in candidates:
        if (item / "best_params_and_alignment.csv").exists():
            return item
    return candidates[0]


def _select_stage6_selector_row(
    *,
    results_root,
    motion_type,
    target_scope,
    adaptive_filter,
    cascade_scheme,
    adaptive_data_type="",
    tw_f=None,
):
    motion_dir = _stage6_motion_dir(results_root, motion_type)
    params_path = motion_dir / "best_params_and_alignment.csv"
    if not params_path.exists():
        raise FileNotFoundError(params_path)
    df = pd.read_csv(params_path)
    candidates = df.copy()
    filters = {
        "motion_type": motion_type,
        "target_scope": target_scope,
        "adaptive_filter": adaptive_filter,
        "cascade_scheme": cascade_scheme,
        "adaptive_data_type": adaptive_data_type,
    }
    for column, expected in filters.items():
        if not expected or column not in candidates.columns:
            continue
        narrowed = candidates[candidates[column].astype(str) == str(expected)]
        if not narrowed.empty:
            candidates = narrowed
    if tw_f is not None and "TW_F" in candidates.columns:
        values = pd.to_numeric(candidates["TW_F"], errors="coerce")
        narrowed = candidates[np.isclose(values.astype(float), float(tw_f), equal_nan=False)]
        if not narrowed.empty:
            candidates = narrowed
    if candidates.empty:
        raise ValueError(f"未找到匹配 Stage-6 参数行: {params_path}")
    return candidates.iloc[0], params_path


if RUN_STAGE7_REPLAY:
    try:
        selector_row, selector_path = _select_stage6_selector_row(
            results_root=REPLAY_RESULTS_ROOT,
            motion_type=REPLAY_MOTION_TYPE,
            target_scope=REPLAY_TARGET_SCOPE,
            adaptive_filter=REPLAY_ADAPTIVE_FILTER,
            cascade_scheme=REPLAY_CASCADE_SCHEME,
            adaptive_data_type=REPLAY_ADAPTIVE_DATA_TYPE,
            tw_f=REPLAY_TW_F,
        )
        REPLAY_SPLIT = REPLAY_MANUAL_SPLIT or str(selector_row.get("split", ""))
        REPLAY_MODE = REPLAY_MANUAL_MODE or str(selector_row.get("mode", ""))
        REPLAY_TARGET_SCOPE = str(selector_row.get("target_scope", REPLAY_TARGET_SCOPE))
        REPLAY_ADAPTIVE_FILTER = str(selector_row.get("adaptive_filter", REPLAY_ADAPTIVE_FILTER))
        REPLAY_ADAPTIVE_DATA_TYPE = str(selector_row.get("adaptive_data_type", REPLAY_ADAPTIVE_DATA_TYPE))
        REPLAY_CASCADE_SCHEME = str(selector_row.get("cascade_scheme", REPLAY_CASCADE_SCHEME))
        print("Stage-7 selector source:", selector_path)
        display(pd.DataFrame([selector_row.to_dict()]))

        replay_paths = replay_best_record_hr_curves(
            signal_csv=REPLAY_SIGNAL_CSV,
            ref_csv=REPLAY_REF_CSV,
            results_root=REPLAY_RESULTS_ROOT,
            output_dir=REPLAY_OUTPUT_DIR,
            motion_type=REPLAY_MOTION_TYPE,
            split=REPLAY_SPLIT,
            mode=REPLAY_MODE,
            target_scope=REPLAY_TARGET_SCOPE,
            adaptive_filter=REPLAY_ADAPTIVE_FILTER,
            adaptive_data_type=REPLAY_ADAPTIVE_DATA_TYPE,
            cascade_scheme=REPLAY_CASCADE_SCHEME,
            TW_F=REPLAY_TW_F,
            fs_origin=FS_ORIGIN,
        )
        REPLAY_CSV = replay_paths["csv"]
        REPLAY_PNG = replay_paths["plot"]
        print("replay CSV:", REPLAY_CSV)
        print("replay PNG:", REPLAY_PNG)
        display(pd.read_csv(REPLAY_CSV).head())
        display(Image(filename=str(REPLAY_PNG)))
    except Exception as exc:
        print("Stage-7 replay 未执行:", exc)
else:
    print("RUN_STAGE7_REPLAY=False，已跳过 Stage-7 replay。")


## 11. 对单个文件的窗口级波形与频谱诊断图重绘

使用 Stage-8 `plot_window_diagnostics_from_records` 重绘单个窗口的原始/滤波波形和频谱诊断图，并展示级联 stage 摘要。


In [ ]:
RUN_WINDOW_DIAGNOSTICS = True

DIAGNOSTIC_OUTPUT_DIR = OUTPUT_ROOT / "window_diagnostics"
MANUAL_ALIGNED_FFT_START_S = 100


def _auto_aligned_fft_start_from_replay(replay_csv):
    if replay_csv is None or not Path(replay_csv).exists():
        return float("nan")
    replay_df = pd.read_csv(replay_csv)
    for column in ["fft_start_s", "time_s"]:
        if column not in replay_df.columns:
            continue
        values = pd.to_numeric(replay_df[column], errors="coerce").dropna()
        values = values[np.isfinite(values)]
        if not values.empty:
            return float(values.iloc[0])
    return float("nan")


if RUN_WINDOW_DIAGNOSTICS:
    aligned_fft_start_s = MANUAL_ALIGNED_FFT_START_S
    if aligned_fft_start_s is None:
        aligned_fft_start_s = _auto_aligned_fft_start_from_replay(globals().get("REPLAY_CSV", None))
    if aligned_fft_start_s is None or not np.isfinite(float(aligned_fft_start_s)):
        aligned_fft_start_s = 0.0
        print("未能自动取得有限 aligned_fft_start_s，暂用 0.0；正式诊断时请手动设置 MANUAL_ALIGNED_FFT_START_S。")
    print("aligned_fft_start_s:", aligned_fft_start_s)

    try:
        diagnostic_paths = plot_window_diagnostics_from_records(
            signal_csv=REPLAY_SIGNAL_CSV,
            ref_csv=REPLAY_REF_CSV,
            results_root=REPLAY_RESULTS_ROOT,
            output_dir=DIAGNOSTIC_OUTPUT_DIR,
            motion_type=REPLAY_MOTION_TYPE,
            split=globals().get("REPLAY_SPLIT", ""),
            mode=globals().get("REPLAY_MODE", ""),
            target_scope=REPLAY_TARGET_SCOPE,
            adaptive_filter=REPLAY_ADAPTIVE_FILTER,
            adaptive_data_type=REPLAY_ADAPTIVE_DATA_TYPE,
            cascade_scheme=REPLAY_CASCADE_SCHEME,
            TW_F=REPLAY_TW_F,
            aligned_fft_start_s=float(aligned_fft_start_s),
            fs_origin=FS_ORIGIN,
        )
        print("waveform:", diagnostic_paths["waveform"])
        print("spectrum:", diagnostic_paths["spectrum"])
        display(Image(filename=str(diagnostic_paths["waveform"])))
        display(Image(filename=str(diagnostic_paths["spectrum"])))

        stages = diagnostic_paths.get("stages", [])
        print("stages non-empty:", bool(stages), "stage_count:", len(stages))
        stage_summary = []
        for idx, stage in enumerate(stages, start=1):
            ranking = stage.get("reference_channel_ranking", "")
            if isinstance(ranking, dict):
                ranking = json.dumps({k: v[:3] for k, v in ranking.items()}, ensure_ascii=False)
            stage_summary.append(
                {
                    "stage": idx,
                    "channel": stage.get("channel", ""),
                    "M": stage.get("M", ""),
                    "K": stage.get("K", ""),
                    "mu": stage.get("mu", stage.get("mu1", "")),
                    "penalty_ref_channel": stage.get("penalty_ref_channel", ""),
                    "reference_channel_ranking": ranking,
                }
            )
        display(pd.DataFrame(stage_summary))
    except Exception as exc:
        print("Stage-8 window diagnostics 未执行:", exc)
else:
    print("RUN_WINDOW_DIAGNOSTICS=False，已跳过 Stage-8 window diagnostics。")


## 12. 对特定滤波器与滤波数据类型的跨运动类型汇总表重绘

使用 Stage-9 `build_cross_motion_summary_table` 汇总同一滤波器、数据类型、target scope 和 `TW_F` 下的跨运动类型指标。


In [ ]:
RUN_CROSS_MOTION_SUMMARY = True

SUMMARY_RESULTS_ROOT = Path(getattr(globals().get("RESULT_TO_CHECK", None), "output_root", RUN_OUTPUT_DIR))
SUMMARY_TABLE_OUTPUT_DIR = OUTPUT_ROOT / "summary_tables"
SUMMARY_ADAPTIVE_FILTER = "lms"
SUMMARY_CASCADE_SCHEME = globals().get("REPLAY_CASCADE_SCHEME", "ACC3")
SUMMARY_ADAPTIVE_DATA_TYPE = ""
SUMMARY_TARGET_SCOPE = globals().get("REPLAY_TARGET_SCOPE", ACTIVE_TARGET_SCOPES[0] if ACTIVE_TARGET_SCOPES else "global")
SUMMARY_TW_F = TW_F

SUMMARY_REQUIRED_COLUMNS = [
    "motion_type",
    "target_scope",
    "split",
    "mode",
    "TW",
    "TW_F",
    "baseline_aae",
    "adaptive_aae",
    "final_aae",
    "posthoc_final_aae",
    "baseline_acc",
    "adaptive_acc",
    "final_acc",
    "posthoc_final_acc",
    "best_tdelay_s",
    "time_bias_after_s",
    "motion_frequency_hz",
    "penalty_ref_channel",
    "final_source_distribution",
]

if RUN_CROSS_MOTION_SUMMARY:
    try:
        summary_path = build_cross_motion_summary_table(
            results_root=SUMMARY_RESULTS_ROOT,
            table_output_dir=SUMMARY_TABLE_OUTPUT_DIR,
            adaptive_filter=SUMMARY_ADAPTIVE_FILTER,
            adaptive_data_type=SUMMARY_ADAPTIVE_DATA_TYPE,
            cascade_scheme=SUMMARY_CASCADE_SCHEME,
            target_scope=SUMMARY_TARGET_SCOPE,
            TW_F=SUMMARY_TW_F,
        )
        print("summary CSV:", summary_path)
        summary_df = pd.read_csv(summary_path)
        display(summary_df.head())
        missing_summary_columns = [column for column in SUMMARY_REQUIRED_COLUMNS if column not in summary_df.columns]
        if missing_summary_columns:
            print("summary 缺少列:", missing_summary_columns)
        else:
            print("summary 必需列检查 OK")
    except Exception as exc:
        print("Stage-9 cross-motion summary 未执行:", exc)
else:
    print("RUN_CROSS_MOTION_SUMMARY=False，已跳过 Stage-9 cross-motion summary。")
